# AI 리뷰 답변 도우미

소상공인 가게의 고객 리뷰를 자동으로 분석하고 전문적인 한국어 답변을 생성하는 시스템입니다.

**파이프라인:**
1. **감성 분류** — multilingual DistilBERT
2. **리뷰 유형 분류** — multilingual SentenceTransformer + cosine similarity
3. **답변 생성** — Qwen2.5-7B-Instruct (4-bit 양자화)

> **실행 순서:** 셀을 위에서 아래로 순서대로 실행하세요.  
> **런타임:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 저장

In [ ]:
# 셀 1 — 패키지 설치 (설치 완료 후 런타임 재시작 필요)
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate sentence-transformers scikit-learn gradio

In [ ]:
# 셀 2 — import + 데이터 클래스
import numpy as np
import gradio as gr
from dataclasses import dataclass
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
class ReviewAnalysis:
    text: str
    sentiment: str
    sentiment_confidence: float
    review_type: str
    type_confidence: float
    strategy: str


@dataclass
class ReplyResult:
    analysis: ReviewAnalysis
    reply: str
    shop_name: str
    business_type: str


print("Import 완료")

In [ ]:
# 셀 3 — ReviewClassifier
class ReviewClassifier:

    CATEGORIES = {
        "delivery_issue":    "배달이 예상 시간보다 늦게 도착하거나, 음식이 식거나 쏟아지거나, 잘못된 메뉴가 오거나, 포장이 훼손되거나, 배달 중 연락이 안 되는 경우",
        "food_quality":      "음식 맛이 없거나, 너무 짜거나 싱겁거나, 양이 가격 대비 적거나, 재료가 신선하지 않거나, 조리가 덜 됐거나, 사진과 실물 차이가 큰 경우",
        "service_complaint": "직원이 불친절하거나 무시하거나, 주문·서빙 대기가 지나치게 길거나, 요청을 무시하거나, 전화·현장 응대 태도가 불량한 경우",
        "hygiene_complaint": "테이블·바닥·화장실 등 매장 청결이 불량하거나, 음식에 이물질이 나오거나, 조리 환경·용기·도구의 위생이 의심되는 경우",
        "unverified_claim":  "식중독·배탈 등 근거 없는 건강 피해 주장이나, 업주가 사실로 확인하기 어려운 과장·허위 내용이 포함된 리뷰",
        "compliment":        "음식 맛·양·가격·서비스·분위기·청결 중 하나 이상을 만족스럽게 평가하며, 재방문 또는 주변 추천 의사를 밝힌 긍정적인 리뷰",
    }

    STRATEGIES = {
        "delivery_issue":    "배달 지연·포장 불량·오배송 등 구체적인 문제를 인정하고 사과한 뒤, 원인 점검과 재발 방지 조치를 약속합니다.",
        "food_quality":      "맛·양·신선도·조리 상태 등 지적된 품질 문제를 겸허히 수용하고, 구체적인 개선 방향을 제시하며 재방문을 유도합니다.",
        "service_complaint": "직원 응대·대기 시간 등 불편한 경험에 진심으로 사과하고, 직원 교육 강화 및 운영 방식 개선을 구체적으로 약속합니다.",
        "hygiene_complaint": "청결 문제·이물질 등 위생 불편에 진심으로 사과하고, 즉각적인 위생 점검과 재발 방지 대책을 구체적으로 약속합니다.",
        "unverified_claim":  "감정적으로 대응하지 않고, 저희 측 위생·운영 기준을 차분하고 정중하게 설명하며 직접 연락을 통한 확인을 안내합니다.",
        "compliment":        "언급된 구체적인 칭찬 항목에 화답하며 진심으로 감사를 표하고, 동일한 품질을 유지하겠다고 약속하며 재방문을 환영합니다.",
        "multi_complaint":   "리뷰에 언급된 각 불만(배달·음식·서비스·위생 등)을 하나씩 구체적으로 인정하고 사과한 뒤, 전 영역에 걸친 개선 의지를 밝힙니다.",
    }

    def __init__(self):
        print("Loading sentiment classifier...")
        self.sentiment_pipe = pipeline(
            task="text-classification",
            model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
            top_k=None,
        )
        print("Loading sentence embedder...")
        self.embedder = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
        )
        self.category_keys = list(self.CATEGORIES.keys())
        self.category_vecs = self.embedder.encode(list(self.CATEGORIES.values()))
        print("Classifier ready.")

    def get_sentiment(self, text):
        output = self.sentiment_pipe(text)[0]
        scores = {r["label"]: r["score"] for r in output}
        sentiment = max(scores, key=scores.get)
        return sentiment, scores[sentiment]

    def get_review_type(self, text):
        vec = self.embedder.encode(text)
        sims = cosine_similarity(vec.reshape(1, -1), self.category_vecs)[0]
        sorted_idx = np.argsort(sims)[::-1]
        top1_key = self.category_keys[sorted_idx[0]]
        top2_key = self.category_keys[sorted_idx[1]]
        top2_sim = float(sims[sorted_idx[1]])
        complaint_labels = {"delivery_issue", "food_quality", "service_complaint", "hygiene_complaint"}
        if top2_sim >= 0.44 and top1_key in complaint_labels and top2_key in complaint_labels:
            return "multi_complaint", float(sims[sorted_idx[0]])
        return top1_key, float(sims[sorted_idx[0]])

    def analyze(self, text):
        sentiment, sent_conf = self.get_sentiment(text)
        review_type, type_conf = self.get_review_type(text)
        negative_types = {
            "delivery_issue", "food_quality", "service_complaint",
            "hygiene_complaint", "unverified_claim", "multi_complaint",
        }
        if sentiment == "positive" and type_conf < 0.45 and review_type not in negative_types:
            review_type = "compliment"
            type_conf = sent_conf
        return ReviewAnalysis(
            text=text,
            sentiment=sentiment,
            sentiment_confidence=sent_conf,
            review_type=review_type,
            type_confidence=type_conf,
            strategy=self.STRATEGIES[review_type],
        )


print("ReviewClassifier 정의 완료")

In [ ]:
# 셀 4 — ReplyGenerator
class ReplyGenerator:

    FEW_SHOT = {
        "delivery_issue": (
            "예시\n"
            "리뷰: \"음식이 1시간이나 늦게 왔고 완전히 식어있었어요. 중간에 연락도 없어서 더 답답했습니다.\"\n"
            "답변: \"고객님, 배달이 1시간이나 지연되고 음식이 식은 채로 전달된 점, 그리고 중간에 아무런 안내도 드리지 못한 점 진심으로 깊이 사과드립니다. "
            "기다리시는 내내 얼마나 답답하고 불편하셨을지 충분히 이해하며, 저희의 배달 운영에 명백한 문제가 있었음을 인정합니다. "
            "해당 시간대 배달 현황을 면밀히 점검하고, 지연 발생 시 즉시 안내 연락을 드리는 체계를 즉각 강화하겠습니다. "
            "다시 한번 불편을 드린 점 사과드리며, 다음에 기회를 주신다면 반드시 더 나은 경험으로 보답드리겠습니다.\""
        ),
        "food_quality": (
            "예시\n"
            "리뷰: \"음식이 너무 짜고 양도 생각보다 훨씬 적었어요. 가격을 생각하면 많이 아쉬웠습니다.\"\n"
            "답변: \"고객님, 음식의 간이 지나치게 짜고 양이 가격에 비해 부족하게 느껴지셨다니 진심으로 죄송합니다. "
            "만족스러운 한 끼를 드실 수 있도록 하는 것이 저희의 기본 책임인데 그러지 못하여 정말 안타깝습니다. "
            "조리 담당자와 즉시 이 내용을 공유하고, 나트륨 조절 기준과 제공량을 전반적으로 재검토하겠습니다. "
            "소중한 피드백을 남겨주셔서 감사드리며, 개선된 모습으로 다시 찾아주실 때 더 나은 경험을 드릴 수 있도록 최선을 다하겠습니다.\""
        ),
        "service_complaint": (
            "예시\n"
            "리뷰: \"직원이 너무 불친절하고 질문해도 제대로 대답을 안 해줬어요. 기분이 많이 상했습니다.\"\n"
            "답변: \"고객님, 저희 직원의 불친절한 응대와 부실한 안내로 기분이 상하셨을 점, 진심으로 깊이 사과드립니다. "
            "방문해 주신 모든 고객님께 따뜻하고 성의 있는 응대를 드리는 것이 저희의 기본 소명인데, 그 역할을 다하지 못하여 정말 부끄럽습니다. "
            "해당 직원과 즉시 면담하고, 전 직원을 대상으로 고객 응대 교육을 강화하겠습니다. "
            "고객님의 소중한 말씀을 계기로 서비스 전반을 돌아보고 반드시 개선된 모습을 보여드리겠습니다.\""
        ),
        "hygiene_complaint": (
            "예시\n"
            "리뷰: \"테이블이 끈적거리고 바닥에 음식물이 그대로 있었어요. 화장실도 너무 지저분했습니다.\"\n"
            "답변: \"고객님, 테이블과 바닥, 화장실 청결 상태가 불량하여 불쾌한 경험을 드린 점 진심으로 깊이 사과드립니다. "
            "청결은 식당이 갖춰야 할 가장 기본적인 조건인데, 이를 지키지 못하여 고객님께 큰 실망을 드렸습니다. "
            "매장 전체 청소 기준을 즉시 강화하고, 영업 중 수시 청결 점검을 의무화하여 재발 방지에 최선을 다하겠습니다. "
            "소중한 말씀 감사드리며, 다음에 방문해 주실 때는 언제나 쾌적한 환경에서 편안하게 식사하실 수 있도록 하겠습니다.\""
        ),
        "unverified_claim": (
            "예시\n"
            "리뷰: \"여기서 먹고 식중독에 걸렸어요. 위생 관리가 전혀 안 되는 것 같습니다.\"\n"
            "답변: \"고객님, 식사 후 몸이 불편하셨다니 우선 걱정이 되고 진심으로 유감스럽습니다. "
            "저희 가게는 매일 위생 점검을 실시하고 식품 안전 기준을 철저히 준수하고 있어, 현재까지 동일한 문제가 접수된 사례가 없음을 말씀드립니다. "
            "다만 고객님께서 불편을 겪으신 만큼, 해당 날짜 조리 과정 전반을 다시 한번 면밀히 확인하겠습니다. "
            "증상이 지속되신다면 언제든지 직접 연락 주시면 성심껏 확인해 드리겠습니다.\""
        ),
        "compliment": (
            "예시\n"
            "리뷰: \"음식도 정말 맛있고 사장님도 너무 친절하셨어요! 분위기도 좋아서 완전 만족합니다. 또 올게요!\"\n"
            "답변: \"고객님, 이렇게 따뜻하고 소중한 말씀을 남겨주셔서 진심으로 감사드립니다. "
            "음식이 맛있으셨고 저희 서비스와 분위기까지 마음에 들어 하셨다니, 저희에게 이보다 큰 보람은 없을 것 같습니다. "
            "앞으로도 고객님이 항상 기분 좋게 드실 수 있도록 음식 품질과 서비스를 한결같이 유지하겠습니다. "
            "다음 방문 때도 따뜻하게 맞이할 수 있도록 최선을 다하겠습니다. 감사합니다!\""
        ),
        "multi_complaint": (
            "예시\n"
            "리뷰: \"배달이 40분이나 늦었고, 음식도 맛이 없고 양도 적었어요. 게다가 직원도 불친절해서 정말 실망했습니다.\"\n"
            "답변: \"고객님, 배달이 40분이나 지연된 점, 음식 맛과 양이 기대에 미치지 못한 점, "
            "그리고 직원 응대가 불쾌하셨던 점까지 여러 부분에서 실망을 드려 진심으로 깊이 사과드립니다. "
            "하나하나가 모두 저희가 반드시 개선해야 할 중요한 사안이며, 이번 경험을 저희 운영 전반을 돌아보는 계기로 삼겠습니다. "
            "배달 시스템 개선, 음식 품질 재검토, 직원 서비스 교육 강화를 즉시 시행하겠습니다. "
            "소중한 의견을 남겨주셔서 감사드리며, 언젠가 다시 찾아주신다면 달라진 모습으로 보답드릴 것을 약속드립니다.\""
        ),
    }

    def __init__(self):
        print("Loading text generation model (Qwen2.5-7B-Instruct, 4-bit)...")
        from transformers import BitsAndBytesConfig
        import torch
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.pipe = pipeline(
            "text-generation",
            model="Qwen/Qwen2.5-7B-Instruct",
            model_kwargs={"quantization_config": bnb_config},
            device_map="auto",
        )
        print("Generator ready.")

    def generate(self, analysis, shop_name, business_type):
        system_prompt = (
            "당신은 \"" + shop_name + "\" (" + business_type + ")의 전문 고객 응대 담당자입니다. "
            "아래 규칙을 반드시 지켜 답변을 작성하세요.\n"
            "1. 리뷰에서 언급된 모든 불만 사항을 구체적으로 인정하고 사과하세요.\n"
            "2. 반드시 한국어로만 답변하세요. 영어, 중국어, 일본어 등 다른 언어는 절대 사용하지 마세요.\n"
            "3. 3~4문장으로 충분히 성의 있게 작성하세요.\n"
            "4. \"다시 방문해 주셔서 죄송합니다\"처럼 의미가 이상한 문장은 절대 사용하지 마세요.\n"
            "5. 고객의 부정적인 표현을 그대로 반복하거나 동조하지 마세요."
        )
        user_prompt = (
            self.FEW_SHOT[analysis.review_type] + "\n\n"
            "이제 아래 리뷰에 대해 같은 스타일로 답변을 작성해 주세요.\n"
            "리뷰에 여러 불만이 있다면 모두 언급하세요.\n"
            "리뷰 유형 : " + analysis.review_type.replace("_", " ") + "\n"
            "답변 전략  : " + analysis.strategy + "\n"
            "리뷰       : \"" + analysis.text + "\"\n"
            "답변:"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
        for attempt in range(2):
            output = self.pipe(
                messages,
                do_sample=True,
                temperature=0.3 if attempt == 0 else 0.1,
                max_new_tokens=200,
            )
            generated = output[0]["generated_text"]
            reply = generated[-1]["content"].strip() if isinstance(generated, list) else generated.strip()
            korean_chars = sum(1 for c in reply if "\uAC00" <= c <= "\uD7A3")
            if len(reply) > 0 and korean_chars / len(reply) >= 0.3:
                return reply
        return reply


print("ReplyGenerator 정의 완료")

In [ ]:
# 셀 5 — ReviewReplyAssistant + evaluate + build_ui
class ReviewReplyAssistant:

    def __init__(self):
        self.classifier = ReviewClassifier()
        self.generator = ReplyGenerator()

    def run(self, review_text, shop_name="우리 가게", business_type="식당", verbose=True):
        analysis = self.classifier.analyze(review_text)
        reply = self.generator.generate(analysis, shop_name, business_type)
        result = ReplyResult(
            analysis=analysis,
            reply=reply,
            shop_name=shop_name,
            business_type=business_type,
        )
        if verbose:
            self._print(result)
        return result

    def _print(self, r):
        a = r.analysis
        print("=" * 62)
        print("리뷰       :", a.text)
        print("가게       :", r.shop_name, " (", r.business_type, ")")
        print("-" * 62)
        print("감정       :", a.sentiment.upper(), "({:.0%})".format(a.sentiment_confidence))
        print("리뷰 유형  :", a.review_type.replace("_", " "), "({:.0%})".format(a.type_confidence))
        print("전략       :", a.strategy)
        print("-" * 62)
        print("답변:\n", r.reply)
        print("=" * 62)


def evaluate(assistant):
    test_cases = [
        {"review": "음식이 한 시간이나 늦게 왔고 완전히 식어있었어요.",     "expected": "delivery_issue"},
        {"review": "햄버거가 눅눅하고 간이 거의 없었어요.",                  "expected": "food_quality"},
        {"review": "직원이 20분 동안 우리를 무시하고 너무 불친절했어요.",    "expected": "service_complaint"},
        {"review": "여기 음식 진짜 최고예요! 이 동네 최고의 카페입니다.",    "expected": "compliment"},
        {"review": "이 식당에서 먹고 식중독에 걸렸어요.",                    "expected": "unverified_claim"},
    ]
    correct = 0
    print("\n분류 평가")
    print("-" * 55)
    for case in test_cases:
        result = assistant.run(case["review"], verbose=False)
        predicted = result.analysis.review_type
        expected = case["expected"]
        mark = "O" if predicted == expected else "X"
        if predicted == expected:
            correct += 1
        print(mark, " 예상:", expected, "/ 결과:", predicted)
    print("-" * 55)
    print("정확도:", correct, "/", len(test_cases), "=", "{:.0%}".format(correct / len(test_cases)))


def build_ui(assistant):
    def process(review_text, shop_name, business_type):
        if not review_text.strip():
            return "리뷰를 입력해 주세요.", "", "", ""
        result = assistant.run(review_text, shop_name, business_type, verbose=False)
        a = result.analysis
        return (
            result.reply,
            a.sentiment.upper() + "  ({:.0%})".format(a.sentiment_confidence),
            a.review_type.replace("_", " ").title() + "  ({:.0%})".format(a.type_confidence),
            a.strategy,
        )

    with gr.Blocks(title="리뷰 답변 도우미", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 리뷰 답변 도우미")
        gr.Markdown("고객 리뷰를 붙여넣으면 즉시 전문적인 한국어 답변을 생성해 드립니다.")
        with gr.Row():
            with gr.Column():
                shop_input = gr.Textbox(label="가게 이름", value="Jenny's Kitchen")
                type_input = gr.Textbox(label="업종", value="식당")
                review_input = gr.Textbox(
                    label="고객 리뷰", lines=5,
                    placeholder="고객 리뷰를 여기에 붙여넣으세요...",
                )
                submit_btn = gr.Button("답변 생성", variant="primary")
            with gr.Column():
                reply_out = gr.Textbox(label="생성된 답변", lines=5)
                sentiment_out = gr.Textbox(label="감정 분석")
                type_out = gr.Textbox(label="리뷰 유형")
                strategy_out = gr.Textbox(label="답변 전략", lines=2)
        gr.Examples(
            examples=[
                ["음식이 30분이나 늦게 왔고 식어있었어요. 다시는 안 시킬게요.", "Jenny's Kitchen", "식당"],
                ["음식도 맛있고 사장님도 너무 친절하셨어요! 또 올게요.",        "Jenny's Kitchen", "식당"],
                ["여기서 먹고 배탈이 났어요. 위생이 걱정됩니다.",               "Jenny's Kitchen", "식당"],
            ],
            inputs=[review_input, shop_input, type_input],
        )
        submit_btn.click(
            fn=process,
            inputs=[review_input, shop_input, type_input],
            outputs=[reply_out, sentiment_out, type_out, strategy_out],
        )
    return demo


print("모든 클래스/함수 정의 완료")

In [ ]:
# 셀 6 — 모델 로드 및 분류 평가
assistant = ReviewReplyAssistant()
evaluate(assistant)

In [ ]:
# 셀 7 — Gradio UI 실행 (share=True 로 공개 URL 생성)
demo = build_ui(assistant)
demo.launch(share=True)